In [1]:
import pandas as pd

In [2]:
import pyspark
from pyspark.sql import SparkSession

In [3]:
from pyspark.sql import types as t
from pyspark.sql import functions as F

In [4]:
spark = SparkSession.builder \
            .master('local[*]') \
            .appName('test') \
            .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/14 05:19:42 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [5]:
!wc -l fhvhv_tripdata_2021-01.csv

11908469 fhvhv_tripdata_2021-01.csv


In [6]:
df = spark.read \
        .option('header', 'true') \
        .csv('fhvhv_tripdata_2021-01.csv')

In [7]:
df.head(5)

[Row(hvfhs_license_num='HV0003', dispatching_base_num='B02682', pickup_datetime='2021-01-01 00:33:44', dropoff_datetime='2021-01-01 00:49:07', PULocationID='230', DOLocationID='166', SR_Flag=None),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02682', pickup_datetime='2021-01-01 00:55:19', dropoff_datetime='2021-01-01 01:18:21', PULocationID='152', DOLocationID='167', SR_Flag=None),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02764', pickup_datetime='2021-01-01 00:23:56', dropoff_datetime='2021-01-01 00:38:05', PULocationID='233', DOLocationID='142', SR_Flag=None),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02764', pickup_datetime='2021-01-01 00:42:51', dropoff_datetime='2021-01-01 00:45:50', PULocationID='142', DOLocationID='143', SR_Flag=None),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02764', pickup_datetime='2021-01-01 00:48:14', dropoff_datetime='2021-01-01 01:08:42', PULocationID='143', DOLocationID='78', SR_Flag=None)]

In [8]:
print(df.schema)

StructType([StructField('hvfhs_license_num', StringType(), True), StructField('dispatching_base_num', StringType(), True), StructField('pickup_datetime', StringType(), True), StructField('dropoff_datetime', StringType(), True), StructField('PULocationID', StringType(), True), StructField('DOLocationID', StringType(), True), StructField('SR_Flag', StringType(), True)])


In [9]:
!head -n 1001 fhvhv_tripdata_2021-01.csv > head.csv

In [10]:
!wc -l head.csv

1001 head.csv


In [11]:
df_pandas = pd.read_csv('head.csv')
df_pandas.head()

,hvfhs_license_num,dispatching_base_num,pickup_datetime,dropoff_datetime,PULocationID,DOLocationID,SR_Flag
0,HV0003,B02682,2021-01-01 00:33:44,2021-01-01 00:49:07,230,166,NaN
1,HV0003,B02682,2021-01-01 00:55:19,2021-01-01 01:18:21,152,167,NaN
2,HV0003,B02764,2021-01-01 00:23:56,2021-01-01 00:38:05,233,142,NaN
3,HV0003,B02764,2021-01-01 00:42:51,2021-01-01 00:45:50,142,143,NaN
4,HV0003,B02764,2021-01-01 00:48:14,2021-01-01 01:08:42,143,78,NaN


In [12]:
df_pandas.dtypes

hvfhs_license_num           str
dispatching_base_num        str
pickup_datetime             str
dropoff_datetime            str
PULocationID              int64
DOLocationID              int64
SR_Flag                 float64
dtype: object

In [13]:
spark.createDataFrame(df_pandas).show()

[Stage 2:>                                                          (0 + 1) / 1]

+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|hvfhs_license_num|dispatching_base_num|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|SR_Flag|
+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|           HV0003|              B02682|2021-01-01 00:33:44|2021-01-01 00:49:07|         230|         166|    NaN|
|           HV0003|              B02682|2021-01-01 00:55:19|2021-01-01 01:18:21|         152|         167|    NaN|
|           HV0003|              B02764|2021-01-01 00:23:56|2021-01-01 00:38:05|         233|         142|    NaN|
|           HV0003|              B02764|2021-01-01 00:42:51|2021-01-01 00:45:50|         142|         143|    NaN|
|           HV0003|              B02764|2021-01-01 00:48:14|2021-01-01 01:08:42|         143|          78|    NaN|
|           HV0005|              B02510|2021-01-01 00:06:59|2021-01-01 00:43:01|

In [14]:
spark.createDataFrame(df_pandas).schema

StructType([StructField('hvfhs_license_num', StringType(), True), StructField('dispatching_base_num', StringType(), True), StructField('pickup_datetime', StringType(), True), StructField('dropoff_datetime', StringType(), True), StructField('PULocationID', LongType(), True), StructField('DOLocationID', LongType(), True), StructField('SR_Flag', DoubleType(), True)])

In [15]:
schema = t.StructType([
    t.StructField('hvfhs_license_num', t.StringType(), True), 
    t.StructField('dispatching_base_num', t.StringType(), True), 
    t.StructField('pickup_datetime', t.TimestampType(), True), 
    t.StructField('dropoff_datetime', t.TimestampType(), True), 
    t.StructField('PULocationID', t.IntegerType(), True), 
    t.StructField('DOLocationID', t.IntegerType(), True), 
    t.StructField('SR_Flag', t.StringType(), True)
])

In [16]:
schema

StructType([StructField('hvfhs_license_num', StringType(), True), StructField('dispatching_base_num', StringType(), True), StructField('pickup_datetime', TimestampType(), True), StructField('dropoff_datetime', TimestampType(), True), StructField('PULocationID', IntegerType(), True), StructField('DOLocationID', IntegerType(), True), StructField('SR_Flag', StringType(), True)])

In [17]:
df = spark.read \
        .option('header', 'true') \
        .schema(schema) \
        .csv('fhvhv_tripdata_2021-01.csv')

In [18]:
df.head(5)

[Row(hvfhs_license_num='HV0003', dispatching_base_num='B02682', pickup_datetime=datetime.datetime(2021, 1, 1, 0, 33, 44), dropoff_datetime=datetime.datetime(2021, 1, 1, 0, 49, 7), PULocationID=230, DOLocationID=166, SR_Flag=None),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02682', pickup_datetime=datetime.datetime(2021, 1, 1, 0, 55, 19), dropoff_datetime=datetime.datetime(2021, 1, 1, 1, 18, 21), PULocationID=152, DOLocationID=167, SR_Flag=None),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02764', pickup_datetime=datetime.datetime(2021, 1, 1, 0, 23, 56), dropoff_datetime=datetime.datetime(2021, 1, 1, 0, 38, 5), PULocationID=233, DOLocationID=142, SR_Flag=None),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02764', pickup_datetime=datetime.datetime(2021, 1, 1, 0, 42, 51), dropoff_datetime=datetime.datetime(2021, 1, 1, 0, 45, 50), PULocationID=142, DOLocationID=143, SR_Flag=None),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02764', pickup_dat

In [19]:
df = df.repartition(24)

In [21]:
df.write.parquet('fhvhv/2021/01', mode='overwrite')

26/03/14 05:20:37 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/03/14 05:20:37 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/03/14 05:20:37 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/03/14 05:20:37 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/03/14 05:20:37 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers
26/03/14 05:20:37 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 58.46% for 13 writers
26/03/14 05:20:37 WARN MemoryManager: Total allocation exceeds 95.

In [22]:
!tree fhvhv/2021/01

fhvhv/2021/01
├── _SUCCESS
├── part-00000-58430099-c1ad-4319-a964-90eb50fd409b-c000.snappy.parquet
├── part-00001-58430099-c1ad-4319-a964-90eb50fd409b-c000.snappy.parquet
├── part-00002-58430099-c1ad-4319-a964-90eb50fd409b-c000.snappy.parquet
├── part-00003-58430099-c1ad-4319-a964-90eb50fd409b-c000.snappy.parquet
├── part-00004-58430099-c1ad-4319-a964-90eb50fd409b-c000.snappy.parquet
├── part-00005-58430099-c1ad-4319-a964-90eb50fd409b-c000.snappy.parquet
├── part-00006-58430099-c1ad-4319-a964-90eb50fd409b-c000.snappy.parquet
├── part-00007-58430099-c1ad-4319-a964-90eb50fd409b-c000.snappy.parquet
├── part-00008-58430099-c1ad-4319-a964-90eb50fd409b-c000.snappy.parquet
├── part-00009-58430099-c1ad-4319-a964-90eb50fd409b-c000.snappy.parquet
├── part-00010-58430099-c1ad-4319-a964-90eb50fd409b-c000.snappy.parquet
├── part-00011-58430099-c1ad-4319-a964-90eb50fd409b-c000.snappy.parquet
├── part-00012-58430099-c1ad-4319-a964-90eb50fd409b-c000.snappy.parquet
├── part-00013-58430099-c1ad-4319-a96

In [23]:
df = spark.read.parquet('fhvhv/2021/01')

In [24]:
df.show()

+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|hvfhs_license_num|dispatching_base_num|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|SR_Flag|
+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|           HV0003|              B02882|2021-01-01 04:07:32|2021-01-01 04:25:29|         167|         179|   NULL|
|           HV0003|              B02765|2021-01-01 15:25:09|2021-01-01 15:37:25|         174|         259|   NULL|
|           HV0004|              B02800|2021-01-01 05:31:50|2021-01-01 05:40:03|         188|          61|   NULL|
|           HV0003|              B02872|2021-01-01 22:10:12|2021-01-01 22:26:42|         129|         129|   NULL|
|           HV0005|              B02510|2021-01-01 03:12:16|2021-01-01 03:21:42|         185|          20|   NULL|
|           HV0005|              B02510|2021-01-02 23:29:58|2021-01-02 23:36:34|

In [25]:
df.printSchema()

root
 |-- hvfhs_license_num: string (nullable = true)
 |-- dispatching_base_num: string (nullable = true)
 |-- pickup_datetime: timestamp (nullable = true)
 |-- dropoff_datetime: timestamp (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- SR_Flag: string (nullable = true)



In [26]:
df.select('pickup_datetime', 'dropoff_datetime', 'PULocationID', 'DOLocationID') \
    .filter(df.hvfhs_license_num == 'HV0003') \
    .show()

+-------------------+-------------------+------------+------------+
|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|
+-------------------+-------------------+------------+------------+
|2021-01-01 04:07:32|2021-01-01 04:25:29|         167|         179|
|2021-01-01 15:25:09|2021-01-01 15:37:25|         174|         259|
|2021-01-01 22:10:12|2021-01-01 22:26:42|         129|         129|
|2021-01-02 14:28:39|2021-01-02 14:33:40|          39|          39|
|2021-01-03 00:56:32|2021-01-03 01:23:02|         254|          41|
|2021-01-02 18:38:55|2021-01-02 18:55:59|          97|         225|
|2021-01-01 04:22:36|2021-01-01 04:42:20|         252|         157|
|2021-01-02 09:56:25|2021-01-02 10:03:31|          42|          42|
|2021-01-01 21:46:04|2021-01-01 22:05:15|         233|         243|
|2021-01-01 01:44:48|2021-01-01 01:57:12|          94|         119|
|2021-01-02 06:30:14|2021-01-02 06:43:01|          31|         119|
|2021-01-01 20:00:52|2021-01-01 20:05:22|       

In [27]:
df \
    .withColumn('pickup_date', F.to_date(df.pickup_datetime)) \
    .withColumn('dropoff_date', F.to_date(df.dropoff_datetime)) \
    .select('pickup_date', 'dropoff_date', 'PULocationID', 'DOLocationID') \
    .show()

+-----------+------------+------------+------------+
|pickup_date|dropoff_date|PULocationID|DOLocationID|
+-----------+------------+------------+------------+
| 2021-01-01|  2021-01-01|         167|         179|
| 2021-01-01|  2021-01-01|         174|         259|
| 2021-01-01|  2021-01-01|         188|          61|
| 2021-01-01|  2021-01-01|         129|         129|
| 2021-01-01|  2021-01-01|         185|          20|
| 2021-01-02|  2021-01-02|         225|          61|
| 2021-01-02|  2021-01-02|          39|          39|
| 2021-01-03|  2021-01-03|         254|          41|
| 2021-01-02|  2021-01-02|          97|         225|
| 2021-01-01|  2021-01-01|          36|          36|
| 2021-01-01|  2021-01-01|         252|         157|
| 2021-01-02|  2021-01-02|          42|          42|
| 2021-01-02|  2021-01-02|         132|         265|
| 2021-01-01|  2021-01-01|         233|         243|
| 2021-01-01|  2021-01-01|          94|         119|
| 2021-01-02|  2021-01-02|          31|       

In [28]:
def crazy_stuff(base_num):
    num = int(base_num[1:])
    if num % 7 == 0:
        return f's/{num:03x}'
    elif num % 3 == 0:
        return f'a/{num:03x}'
    else:
        return f'e/{num:03x}'

In [29]:
crazy_stuff('B02882')

'e/b42'

In [30]:
crazy_stuff_udf = F.udf(crazy_stuff, returnType=t.StringType())

In [31]:
df \
    .withColumn('pickup_date', F.to_date(df.pickup_datetime)) \
    .withColumn('dropoff_date', F.to_date(df.dropoff_datetime)) \
    .withColumn('base_id', crazy_stuff_udf(df.dispatching_base_num)) \
    .select('base_id', 'pickup_date', 'dropoff_date', 'PULocationID', 'DOLocationID') \
    .show()

[Stage 12:>                                                         (0 + 1) / 1]

+-------+-----------+------------+------------+------------+
|base_id|pickup_date|dropoff_date|PULocationID|DOLocationID|
+-------+-----------+------------+------------+------------+
|  e/b42| 2021-01-01|  2021-01-01|         167|         179|
|  s/acd| 2021-01-01|  2021-01-01|         174|         259|
|  s/af0| 2021-01-01|  2021-01-01|         188|          61|
|  e/b38| 2021-01-01|  2021-01-01|         129|         129|
|  e/9ce| 2021-01-01|  2021-01-01|         185|          20|
|  e/9ce| 2021-01-02|  2021-01-02|         225|          61|
|  e/b3b| 2021-01-02|  2021-01-02|          39|          39|
|  e/b47| 2021-01-03|  2021-01-03|         254|          41|
|  e/b35| 2021-01-02|  2021-01-02|          97|         225|
|  e/9ce| 2021-01-01|  2021-01-01|          36|          36|
|  s/b13| 2021-01-01|  2021-01-01|         252|         157|
|  e/b3e| 2021-01-02|  2021-01-02|          42|          42|
|  e/9ce| 2021-01-02|  2021-01-02|         132|         265|
|  e/acc| 2021-01-01|  2